# Exceptions, Tracebacks, and Control Flow When Things Go Wrong
This notebook has been rewritten to be slower, more reflective, and less template-like. Instead of treating **Exceptions, Tracebacks, and Control Flow When Things Go Wrong** as a small isolated topic, the goal here is to treat it as part of Python's larger runtime story: objects are created, names are bound, frames execute bytecode, and the interpreter keeps following rules whether or not the source code looks simple from the outside.

If this topic ever felt a little too neat or too magical when explained quickly, that is exactly what this rewrite is trying to fix. We are going to keep asking two grounding questions: what changed in memory, and what instructions is the interpreter actually stepping through under the surface?

The point is not to drown in internals. The point is to make the language feel explainable. Once that happens, even advanced behavior starts feeling much less mysterious.


When people struggle with **Exceptions, Tracebacks, and Control Flow When Things Go Wrong**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


- See exceptions as objects and control-flow signals
- Read tracebacks more confidently
- Use try/except/else/finally with intention
- Create custom exceptions when meaning matters


When an exception is raised, Python creates or uses an exception object and starts unwinding the stack until a matching handler is found. That unwinding story is why tracebacks are frame histories.


In [1]:
try:
    1 / 0
except ZeroDivisionError as exc:
    print(type(exc))
    print(exc)


<class 'ZeroDivisionError'>
division by zero


Exception-related bytecode is more involved than a simple arithmetic function, but even a small example reminds us that `try` blocks are translated into real interpreter control-flow machinery.


In [2]:
import dis

def safe_div(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None

dis.dis(safe_div)


  3           0 RESUME                   0

  4           2 NOP

  5           4 LOAD_FAST                0 (a)
              6 LOAD_FAST                1 (b)
              8 BINARY_OP               11 (/)
             12 RETURN_VALUE
        >>   14 PUSH_EXC_INFO

  6          16 LOAD_GLOBAL              0 (ZeroDivisionError)
             28 CHECK_EXC_MATCH
             30 POP_JUMP_FORWARD_IF_FALSE     4 (to 40)
             32 POP_TOP

  7          34 POP_EXCEPT
             36 LOAD_CONST               0 (None)
             38 RETURN_VALUE

  6     >>   40 RERAISE                  0
        >>   42 COPY                     3
             44 POP_EXCEPT
             46 RERAISE                  1
ExceptionTable:
  4 to 10 -> 14 [0]
  14 to 32 -> 42 [1] lasti
  40 to 40 -> 42 [1] lasti


They are not only messages on the screen.

That is why they are so powerful for debugging.

Swallowing all exceptions makes debugging harder, not easier.

They let callers distinguish one problem from another.


The traceback is a breadcrumb trail of how the program got here.


In [3]:
def a():
    b()

def b():
    c()

def c():
    raise RuntimeError("something broke")

try:
    a()
except RuntimeError:
    import traceback
    traceback.print_exc()


Traceback (most recent call last):
  File "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_14740\926151377.py", line 11, in <module>
    a()
  File "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_14740\926151377.py", line 2, in a
    b()
  File "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_14740\926151377.py", line 5, in b
    c()
  File "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_14740\926151377.py", line 8, in c
    raise RuntimeError("something broke")
RuntimeError: something broke


`else` means “no exception happened”. `finally` means “do this regardless”.


In [4]:
try:
    value = int("42")
except ValueError:
    print("bad input")
else:
    print("parsed", value)
finally:
    print("cleanup always runs")


parsed 42
cleanup always runs


Callers can now specifically catch your domain-level error.


In [5]:
class ValidationError(Exception):
    pass

def validate_age(age):
    if age < 0:
        raise ValidationError("age cannot be negative")
    return age

try:
    validate_age(-1)
except ValidationError as exc:
    print(exc)


age cannot be negative


This is not only a classroom topic. **Exceptions, Tracebacks, and Control Flow When Things Go Wrong** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


- Using bare `except:` too freely
- Catching errors you cannot really handle
- Ignoring tracebacks and jumping straight to guesses


- What is an exception object?
- What does `finally` do?
- Why create custom exception classes?


- Exceptions are structured error signals.
- Tracebacks are frame histories.
- Handle only what you can responsibly handle.


If this notebook did its job, **Exceptions, Tracebacks, and Control Flow When Things Go Wrong** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.
